# Добавить правки
1. Исправить опыт в резюме Сейчас experience_text извлекается только если есть и годы, и месяцы


In [262]:
import pandas as pd
import re
import time
import json
import requests
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
import plotly.express as px

## Обработка сырых данных

### Обработка резюме

In [181]:
df = pd.read_csv('data/data_rezume.csv')

In [182]:
df.head()

,id,resume_url,header_text,wrapper_text,created_at,updated_at
0,1,https://hh.ru/resume/3bab3ab0000908964c0039ed1...,"Был сегодня в 14:55\nКандидат\nМужчина, 27 лет...",Аналитик\nСпециализации:\nАналитик\nТип занято...,2026-02-23 17:13:27.709 +0300,2026-02-23 17:13:27.709 +0300
1,2,https://hh.ru/resume/c0d0ac9b0002b4446d0039ed1...,"Был меньше недели назад\nКандидат\nМужчина, 32...",Аналитик данных\nСпециализации:\nАналитик\nТип...,2026-02-23 17:14:55.601 +0300,2026-02-23 17:14:55.601 +0300
2,3,https://hh.ru/resume/a28fcb6700084b93170039ed1...,"Была меньше недели назад\nКандидат\nЖенщина, 3...",Аналитик данных\n300 000 ₽ на руки\nСпециализа...,2026-02-23 17:15:11.186 +0300,2026-02-23 17:15:11.186 +0300
3,4,https://hh.ru/resume/b1b0872200090d4ab70039ed1...,"Был меньше недели назад\nКандидат\nМужчина, 24...",Аналитик данных\n180 000 ₽ на руки\nСпециализа...,2026-02-23 17:15:27.759 +0300,2026-02-23 17:15:27.759 +0300
4,5,https://hh.ru/resume/a0c1974500055dfbf30039ed1...,"Был сегодня в 11:49\nКандидат\nМужчина, 36 лет...",Аналитик\n150 000 ₽ на руки\nСпециализации:\nB...,2026-02-23 17:15:42.736 +0300,2026-02-23 17:15:42.736 +0300


In [183]:
df.shape

(104887, 6)

Из текста карточек резюме извлекаем следующие атрибуты:
- Название вакансии resume_title
- Навыки, указанные в резюме skills_list
- Опыт работы experience_text
- Специализации в резюме specializations_list
- Гражданство citizenship

In [184]:
df['resume_title'] = (
    df['wrapper_text']
    .fillna('')
    .astype(str)
    .str.split('\n')
    .str[0]
    .str.strip()
)

In [185]:
def extract_skills(text):
    text = str(text)
    match = re.search(r'Навыки\s*(.*?)\s*Обо мне', text, flags=re.S)
    if not match:
        return None
    block = match.group(1).strip()
    skills = []
    for line in block.split('\n'):
        line = line.strip()
        if not line:
            continue
        if line == 'Уровни владения навыками':
            continue
        skills.append(line)
    return skills if skills else None

df['skills_list'] = df['wrapper_text'].apply(extract_skills)

In [186]:
def extract_skills(text):
    if pd.isna(text):
        return None
    
    text = str(text)

    start_match = re.search(r'(?m)^Навыки\s*$', text)
    if not start_match:
        return None

    start = start_match.end()

    end_match = re.search(
        r'(?m)^(Обо мне|Опыт вождения|Высшее образование.*|Знание языков|Гражданство, время в пути до работы)\s*$',
        text[start:]
    )

    if end_match:
        block = text[start:start + end_match.start()]
    else:
        block = text[start:]

    lines = []
    skip_lines = {
        'Уровни владения навыками',
        'Продвинутый уровень',
        'Средний уровень',
        'Начальный уровень'
    }

    for line in block.split('\n'):
        line = line.strip()
        if not line:
            continue
        if line in skip_lines:
            continue
        lines.append(line)

    skills = list(dict.fromkeys(lines))

    return skills if skills else None

df['skills_list'] = df['wrapper_text'].apply(extract_skills)

In [187]:
def normalize_skills(skills):
    if not isinstance(skills, list):
        return []
    
    result = []
    seen = set()

    for skill in skills:
        if not isinstance(skill, str):
            continue
        skill = skill.strip().lower()
        if not skill:
            continue
        if skill not in seen:
            seen.add(skill)
            result.append(skill)
    
    return result

df["skills_list"] = df["skills_list"].apply(normalize_skills)

In [188]:
def extract_specializations(text):
    text = str(text)
    match = re.search(
        r'Специализац(?:ия|ии):\s*(.*?)\nТип занятости',
        text,
        flags=re.S
    )
    if not match:
        return None
    block = match.group(1).strip()   
    specializations = []
    for line in block.split('\n'):
        line = line.strip()
        if line:
            specializations.append(line)
    return specializations if specializations else None

df['specializations_list'] = df['wrapper_text'].apply(extract_specializations)

In [189]:
def extract_experience_text(text):
    text = str(text)
    match = re.search(
        r'Опыт работы\s+(\d+)\s+(?:год|года|лет)\s+(\d+)\s+(?:месяц|месяца|месяцев)',
        text
    )
    if match:
        years = int(match.group(1))
        months = int(match.group(2))
        return round(years + months / 12, 2)
    return None

df['experience_text'] = df['wrapper_text'].apply(extract_experience_text)

In [190]:
def extract_citizenship(text):
    text = str(text)
    match = re.search(
        r'Гражданство, время в пути до работы\s*\nГражданство:\s*(.*)',
        text
    )
    if match:
        return match.group(1).strip()
    return None

df['citizenship'] = df['wrapper_text'].apply(extract_citizenship)

#### Отбор релевантных резюме

Так как в датасете встречается очень много нерелевантных резюме (не относящихся к области анализа данных и работы с данными), необходимо оставить только нужные

Для начала оставим только те резюме, в названии которых присутствуют заголовки, используемые в парсинге (без учета регистра, если хотя бы заголовок парсинга просто встречался в заголовке вакансии)

In [191]:
prof_list = [
    'Аналитик DWH',
    'SQL Analyst',
    'Аналитик больших данных',
    'Специалист по анализу данных',
    'Data Scientist',
    'Data Engineer',
    'BI Developer',
    'ML Engineer',
    'Аналитик данных',
    'Data Analyst',
    'BI-аналитик',
    'Продуктовый аналитик'
]

pattern = r'(^|\b)(' + '|'.join(re.escape(x) for x in prof_list) + r')($|\b)'

mask = df['resume_title'].fillna('').str.contains(
    pattern,
    case=False,
    regex=True
)

df_new = df[mask].copy()

C:\Temp\ipykernel_4824\2903095158.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df['resume_title'].fillna('').str.contains(


In [192]:
df_rez = df_new.copy()

#### Обработка текста тела резюме

In [ ]:
def clean_resume_text(text):
    # Удаление до строки "Тип занятости: ..." включительно
    text = re.sub(
        r"^.*?тип занятости\s*:.*?\n",
        " ",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    # Удаление строки "Опыт работы ..." с годами и/или месяцами
    text = re.sub(
        r"опыт работы\s+"
        r"(?:(?:\d+)\s+(?:года|год|лет)\s*)?"
        r"(?:(?:\d+)\s+(?:месяцев|месяца|месяц)\s*)?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Удаление после "Гражданство, время в пути до работы"
    text = re.sub(
        r"гражданство,\s*время в пути до работы.*$",
        " ",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    # Удаление "Показать еще"
    text = re.sub(
        r"показать\s+еще",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Удаление длительности работы: "1 год 6 месяцев", "3 года", "11 месяцев"
    text = re.sub(
        r"\b\d+\s+(?:года|год|лет)\s*(?:\d+\s+(?:месяцев|месяца|месяц))?\b"
        r"|\b\d+\s+(?:месяцев|месяца|месяц)\b",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"знание языков.*?(?=повышение квалификации,\s*курсы|$)",
        " ",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    # базовая очистка
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\b\S+@\S+\.\S+\b", " ", text)
    text = re.sub(r"[^a-zа-яё0-9+#.\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [194]:
# Для теста
# df_rez["text_clean_test"] = df_rez["wrapper_text"].apply(clean_resume_text)

In [195]:
df_rez["text_clean"] = df_rez["wrapper_text"].apply(clean_resume_text)

In [198]:
df_rez.head()

,id,resume_url,header_text,wrapper_text,created_at,updated_at,resume_title,skills_list,specializations_list,experience_text,citizenship,text_clean
1,2,https://hh.ru/resume/c0d0ac9b0002b4446d0039ed1...,"Был меньше недели назад\nКандидат\nМужчина, 32...",Аналитик данных\nСпециализации:\nАналитик\nТип...,2026-02-23 17:14:55.601 +0300,2026-02-23 17:14:55.601 +0300,Аналитик данных,"[пользователь пк, ms sql, ms office, driving l...",[Аналитик],8.92,Россия,сентябрь 2024 сентябрь 2025 кофемания москва г...
2,3,https://hh.ru/resume/a28fcb6700084b93170039ed1...,"Была меньше недели назад\nКандидат\nЖенщина, 3...",Аналитик данных\n300 000 ₽ на руки\nСпециализа...,2026-02-23 17:15:11.186 +0300,2026-02-23 17:15:11.186 +0300,Аналитик данных,"[ms powerpoint, python, numpy, sql, git, pycha...","[BI-аналитик, аналитик данных, Аналитик, Бизне...",6.50,Россия,апрель 2025 по настоящее время termoland услуг...
3,4,https://hh.ru/resume/b1b0872200090d4ab70039ed1...,"Был меньше недели назад\nКандидат\nМужчина, 24...",Аналитик данных\n180 000 ₽ на руки\nСпециализа...,2026-02-23 17:15:27.759 +0300,2026-02-23 17:15:27.759 +0300,Аналитик данных,"[data science, регрессионный анализ, проверка ...",[Аналитик],2.67,Россия,декабрь 2024 по настоящее время центральный ба...
5,6,https://hh.ru/resume/989115340003c2cb260039ed1...,"Был сегодня в 12:23\nКандидат\nМужчина, 35 лет...",Аналитик данных\n150 000 ₽ на руки\nСпециализа...,2026-02-23 17:15:55.201 +0300,2026-02-23 17:15:55.201 +0300,Аналитик данных,"[sql, power bi, ms power bi, dax, tableau, abc...","[BI-аналитик, аналитик данных, Аналитик, Бизне...",NaN,Россия,январь 2025 по настоящее время нева дельта спб...
7,8,https://hh.ru/resume/29c348910002487e500039ed1...,"Был сегодня в 12:54\nКандидат\nМужчина, 39 лет...","Аналитик данных\nСпециализации:\nBI-аналитик, ...",2026-02-23 17:16:25.280 +0300,2026-02-23 17:16:25.280 +0300,Аналитик данных,"[python, vba, sql, ms office, oracle, qlik sen...","[BI-аналитик, аналитик данных]",14.42,Россия,октябрь 2011 по настоящее время сбер москва ra...


In [199]:
df_rez.to_csv("df_rez_new.csv", index=False)

### Обработка вакансий

In [200]:
df = pd.read_csv('data/vacant.csv')

In [201]:
df.head()

,id,vacancy_url,header_text,wrapper_text,created_at,updated_at
0,1,https://hh.ru/vacancy/129575262,Старший разработчик в группу GPU-инфраструктур...,Yandex\nInfrastructure\nМы создаём и развиваем...,2026-03-17 21:54:16.843,2026-03-17 21:54:16.843
1,2,https://hh.ru/vacancy/129454986,Project manager по развитию и трансформации би...,Обязанности:\nВыполнение функций project manag...,2026-03-17 21:54:39.568,2026-03-17 21:54:39.568
2,3,https://hh.ru/vacancy/111241834,QA Engineer (со знанием немецкого языка)\nУров...,Team.Inno – одна из наиболее опытных белорусск...,2026-03-17 21:55:04.099,2026-03-17 21:55:04.099
3,4,https://hh.ru/vacancy/129422503,Менеджер по маркетплейсу OZON\nВ архиве с 11 м...,Привет!\nМеня зовут Максим - я селлер на ВБ и ...,2026-03-17 21:55:33.566,2026-03-17 21:55:33.566
4,5,https://hh.ru/vacancy/129356560,Менеджер по закупкам и снабжению\nВ архиве с 1...,Tonka Perfumes Moscow – российский парфюмерный...,2026-03-17 21:56:04.140,2026-03-17 21:56:04.140


In [202]:
df["header_text_list"] = df["header_text"].fillna("").str.split("\n")

In [203]:
df["header_text_list"] = df["header_text_list"].apply(
    lambda x: [item for item in x if not (isinstance(item, str) and item.startswith("В архиве"))]
    if isinstance(x, list) else x
)

Из текста карточек вакансий извлекаем следующие атрибуты:
 - Наименование vacancy_name
 - Опыт работы experience
 - Указанная вилка по з/п salary
 - Флаг присутствия вилки по з/п salary_exists, от и до salary_from, salary_to
 - Форматт работы work_format
 - Тип занятости employment_type
 - Требуемые навыки skills_list
 - Опыт работы (от скольки лет, числовое представвление) experience_years_min

In [204]:
df["vacancy_name"] = df["header_text_list"].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
)

In [205]:
df["experience"] = df["header_text_list"].apply(
    lambda x: next(
        (
            item.replace("Опыт работы:", "").strip()
            for item in x
            if isinstance(item, str) and item.startswith("Опыт работы:")
        ),
        None
    ) if isinstance(x, list) else None
)

In [206]:
df["salary"] = df["header_text_list"].apply(
    lambda x: x[1] if isinstance(x, list) and len(x) > 1 else None
)

In [207]:
def parse_salary(text):
    if pd.isna(text) or str(text).strip() == "Уровень дохода не указан":
        return pd.Series([False, None, None])

    text = str(text).strip()

    numbers = re.findall(r"\d[\d\s]*", text)
    numbers = [int(num.replace(" ", "")) for num in numbers]

    if len(numbers) == 0:
        return pd.Series([False, None, None])

    if len(numbers) == 1:
        number = numbers[0]

        if text.startswith("от"):
            return pd.Series([True, number, None])

        if text.startswith("до"):
            return pd.Series([True, None, number])

        return pd.Series([True, number, number])

    return pd.Series([True, numbers[0], numbers[1]])

df[["salary_exists", "salary_from", "salary_to"]] = df["salary"].apply(parse_salary)

In [208]:
df["work_format"] = df["header_text_list"].apply(
    lambda x: x[-1].replace("Формат работы:", "").strip()
    if isinstance(x, list) and len(x) > 0 and isinstance(x[-1], str) and x[-1].startswith("Формат работы:")
    else None
)

In [209]:
df["employment_type"] = df["header_text_list"].apply(
    lambda x: next(
        (item for item in x if isinstance(item, str) and "занятость" in item.lower()),
        None
    ) if isinstance(x, list) else None
)

In [210]:
def parse_skills(text):
    if pd.isna(text):
        return []
    
    text = str(text)
    
    match = re.search(
        r"Ключевые навыки\s*(.*?)\s*Где предстоит работать",
        text,
        flags=re.S
    )
    
    if not match:
        return []
    
    skills_block = match.group(1).strip()
    
    skills = [
        line.strip()
        for line in skills_block.split("\n")
        if line.strip()
    ]
    
    return skills

df["skills_list"] = df["wrapper_text"].apply(parse_skills)

In [211]:
def parse_experience(text):
    if pd.isna(text):
        return None

    text = str(text).lower().strip().replace('–', '-')

    if 'нет опыта' in text:
        return 0
    if 'более 6' in text:
        return 6

    match = re.search(r'(\d+)\s*-\s*(\d+)', text)
    if match:
        return int(match.group(1))

    match = re.search(r'от\s*(\d+)', text)
    if match:
        return int(match.group(1))

    return None

df['experience_years_min'] = df['experience'].apply(parse_experience)

#### Отбор релевантных вакансий

Так же, как и с резюме, при парсинге получили большое количество нерелевантных вакансий

Поэтому поступим для начала так же - отберем вакансии по заголовку парсинга

In [212]:
prof_list = [
    'Аналитик DWH',
    'SQL Analyst',
    'Аналитик больших данных',
    'Специалист по анализу данных',
    'Data Scientist',
    'Data Engineer',
    'BI Developer',
    'ML Engineer',
    'Аналитик данных',
    'Data Analyst',
    'BI-аналитик',
    'Продуктовый аналитик'
]

pattern = r'(^|\b)(' + '|'.join(re.escape(x) for x in prof_list) + r')($|\b)'

mask = df['vacancy_name'].fillna('').str.contains(
    pattern,
    case=False,
    regex=True
)

df_new = df[mask].copy()

C:\Temp\ipykernel_4824\3057247236.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df['vacancy_name'].fillna('').str.contains(


In [213]:
df_vac = df_new.copy()

#### Дополнительная обработка навыков с помощью LLM

In [214]:
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "qwen2.5:7b"

In [ ]:
def is_nonempty_list(x):
    return isinstance(x, list) and len(x) > 0

In [ ]:
def remove_key_skills_block(text):
    if pd.isna(text) or not str(text).strip():
        return text
    text = str(text)
    text = re.sub(
        r"Ключевые навыки\s*(.*?)\s*Где предстоит работать",
        " ",
        text,
        flags=re.S
    )
    return text

In [217]:
# Извлечение JSON из ответа модели
def extract_json_object(text):
    try:
        return json.loads(text)
    except:
        pass
    match = re.search(r"\{.*\}", text, flags=re.S)
    if match:
        try:
            return json.loads(match.group(0))
        except:
            return None
    return None

In [218]:
def build_prompt(text):
    text = remove_key_skills_block(text)

    return f"""
Извлеки из текста вакансии только явно указанные профессиональные навыки.

Считать навыками:
- технологии
- инструменты
- языки программирования
- библиотеки
- фреймворки
- базы данных
- BI-инструменты
- методы анализа данных
- платформы и сервисы

Не считать навыками:
- soft skills
- личные качества
- общие фразы вроде "ответственность", "обучаемость", "работа в команде"
- слишком общие слова вроде "анализ", "разработка", "ведение отчетности"

Важно:
- не смотри на блок между "Ключевые навыки" и "Где предстоит работать"
- извлекай навыки из остального текста вакансии

Правила:
1. Не добавляй навыки, которых нет в тексте.
2. Верни только JSON-объект такого вида:
{{"skills": ["skill1", "skill2"]}}
3. Если явных навыков нет, верни:
{{"skills": []}}

Текст вакансии:
{text}
"""

In [219]:
def extract_skills_with_ollama(text, model=OLLAMA_MODEL, timeout=180):
    if pd.isna(text) or not str(text).strip():
        return []

    prompt = build_prompt(text)

    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "format": {
            "type": "object",
            "properties": {
                "skills": {
                    "type": "array",
                    "items": {"type": "string"}
                }
            },
            "required": ["skills"]
        },
        "options": {
            "temperature": 0
        }
    }

    response = requests.post(OLLAMA_URL, json=payload, timeout=timeout)
    response.raise_for_status()

    raw_answer = response.json().get("response", "")
    parsed_answer = extract_json_object(raw_answer)

    if parsed_answer is None:
        return []

    skills = parsed_answer.get("skills", [])

    if not isinstance(skills, list):
        return []

    return skills

In [220]:
if "skills_llm" not in df_vac.columns:
    df_vac["skills_llm"] = None

df_vac["skills_final"] = df_vac["skills_list"].apply(
    lambda x: x if isinstance(x, list) else []
)

In [221]:
mask_no_manual_skills = (
    ~df_vac["skills_list"].apply(is_nonempty_list)
    & df_vac["wrapper_text"].notna()
    & (df_vac["wrapper_text"].str.strip() != "")
)
print("Строк без навыков, найденных вручную:", mask_no_manual_skills.sum())

Строк без навыков, найденных вручную: 865


In [222]:
# LLM для строк, где вручную ничего не нашлось
results_no_manual = []

subset_no_manual = df_vac.loc[mask_no_manual_skills, ["wrapper_text"]].copy()

for i, (idx, text) in enumerate(subset_no_manual["wrapper_text"].items(), start=1):
    try:
        skills = extract_skills_with_ollama(text)
    except Exception as e:
        print(f"Ошибка на индексе {idx}: {e}")
        skills = []

    results_no_manual.append((idx, skills))

    if i % 20 == 0:
        print(f"Обработано {i} из {len(subset_no_manual)}")

Обработано 20 из 865
Обработано 40 из 865
Обработано 60 из 865
Обработано 80 из 865
Обработано 100 из 865
Обработано 120 из 865
Обработано 140 из 865
Обработано 160 из 865
Обработано 180 из 865
Обработано 200 из 865
Обработано 220 из 865
Обработано 240 из 865
Обработано 260 из 865
Обработано 280 из 865
Обработано 300 из 865
Обработано 320 из 865
Обработано 340 из 865
Обработано 360 из 865
Обработано 380 из 865
Обработано 400 из 865
Обработано 420 из 865
Обработано 440 из 865
Обработано 460 из 865
Обработано 480 из 865
Обработано 500 из 865
Обработано 520 из 865
Обработано 540 из 865
Обработано 560 из 865
Обработано 580 из 865
Обработано 600 из 865
Обработано 620 из 865
Обработано 640 из 865
Обработано 660 из 865
Обработано 680 из 865
Обработано 700 из 865
Обработано 720 из 865
Обработано 740 из 865
Обработано 760 из 865
Обработано 780 из 865
Обработано 800 из 865
Обработано 820 из 865
Обработано 840 из 865
Обработано 860 из 865


In [223]:
for idx, skills in results_no_manual:
    df_vac.at[idx, "skills_llm"] = skills
    df_vac.at[idx, "skills_final"] = skills if isinstance(skills, list) else []

In [224]:
mask_manual_skills = (
    df_vac["skills_list"].apply(is_nonempty_list)
    & df_vac["wrapper_text"].notna()
    & (df_vac["wrapper_text"].str.strip() != "")
)

print("Строк с навыками, найденными вручную:", mask_manual_skills.sum())

Строк с навыками, найденными вручную: 600


In [225]:
# LLM для строк, где вручную навыки уже нашлись
results_with_manual = []

subset_with_manual = df_vac.loc[mask_manual_skills, ["wrapper_text"]].copy()

for i, (idx, text) in enumerate(subset_with_manual["wrapper_text"].items(), start=1):
    try:
        skills = extract_skills_with_ollama(text)
    except Exception as e:
        print(f"Ошибка на индексе {idx}: {e}")
        skills = []

    results_with_manual.append((idx, skills))

    if i % 20 == 0:
        print(f"Обработано {i} из {len(subset_with_manual)}")

Обработано 20 из 600
Обработано 40 из 600
Обработано 60 из 600
Обработано 80 из 600
Обработано 100 из 600
Обработано 120 из 600
Обработано 140 из 600
Обработано 160 из 600
Обработано 180 из 600
Обработано 200 из 600
Обработано 220 из 600
Обработано 240 из 600
Обработано 260 из 600
Обработано 280 из 600
Обработано 300 из 600
Обработано 320 из 600
Обработано 340 из 600
Обработано 360 из 600
Обработано 380 из 600
Обработано 400 из 600
Обработано 420 из 600
Обработано 440 из 600
Обработано 460 из 600
Обработано 480 из 600
Обработано 500 из 600
Обработано 520 из 600
Обработано 540 из 600
Обработано 560 из 600
Обработано 580 из 600
Обработано 600 из 600


In [226]:
for idx, skills in results_with_manual:
    df_vac.at[idx, "skills_llm"] = skills

    current_skills = df_vac.at[idx, "skills_final"]
    current_skills = current_skills if isinstance(current_skills, list) else []

    llm_skills = skills if isinstance(skills, list) else []

    merged = current_skills + llm_skills

    final_result = []
    seen = set()

    for skill in merged:
        if not isinstance(skill, str):
            continue

        skill = skill.strip().lower()

        if not skill:
            continue

        if skill not in seen:
            seen.add(skill)
            final_result.append(skill)

    df_vac.at[idx, "skills_final"] = final_result

In [227]:
def lowercase_and_deduplicate(skills):
    if not isinstance(skills, list):
        return []

    result = []
    seen = set()

    for skill in skills:
        if not isinstance(skill, str):
            continue

        skill = skill.strip().lower()

        if not skill:
            continue

        if skill not in seen:
            seen.add(skill)
            result.append(skill)

    return result

In [228]:
df_vac["skills_final"] = df_vac["skills_final"].apply(lowercase_and_deduplicate)

#### Обработка текста тела вакансий

In [229]:
# df_vac = pd.read_csv('df_vac_intermediate.csv')
df_vac.head()

,id,vacancy_url,header_text,wrapper_text,created_at,updated_at,header_text_list,vacancy_name,experience,salary,salary_exists,salary_from,salary_to,work_format,employment_type,skills_list,experience_years_min,skills_llm,skills_final
16,15,https://hh.ru/vacancy/129592277,Аналитик данных / экономист инвестиционных про...,46\nорганов исполнительной власти\n132\nтеррит...,2026-03-17 22:02:49.589,2026-03-17 22:02:49.589,[Аналитик данных / экономист инвестиционных пр...,Аналитик данных / экономист инвестиционных про...,1–3 года,Уровень дохода не указан,False,NaN,NaN,на месте работодателя,Полная занятость,[],1.0,"[Python, Django, Pandas, MS Office (Power Poin...","[python, django, pandas, ms office (power poin..."
34,34,https://hh.ru/vacancy/129853395,Data Engineer (разработчик DWH)\nВ архиве с 17...,Задаем тренды в технологиях ритейла\nX5 Group ...,2026-03-17 22:13:18.663,2026-03-17 22:13:18.663,"[Data Engineer (разработчик DWH), Уровень дохо...",Data Engineer (разработчик DWH),3–6 лет,Уровень дохода не указан,False,NaN,NaN,"на месте работодателя, удалённо или гибрид",Полная занятость,"[SQL, Python, Big Data, Apache Airflow, Apache...",3.0,"[Spark, Flink, ClickHouse, Kafka, Trino, Apach...","[sql, python, big data, apache airflow, apache..."
47,92,https://hh.ru/vacancy/129525525,Аналитик данных\nВ архиве с 15 февраля 2026\nУ...,Центр Компетенций Бизнес аналитики и финансово...,2026-03-17 22:56:00.590,2026-03-17 22:56:00.590,"[Аналитик данных, Уровень дохода не указан, Оп...",Аналитик данных,1–3 года,Уровень дохода не указан,False,NaN,NaN,на месте работодателя,Полная занятость,[],1.0,"[SQL, Python, Hadoop, Excel]","[sql, python, hadoop, excel]"
64,162,https://hh.ru/vacancy/130038332,Data Engineer по построению DWH\nУровень доход...,Компания EcoFinance развивает и внедряет проду...,2026-03-17 23:43:51.117,2026-03-17 23:43:51.117,"[Data Engineer по построению DWH, Уровень дохо...",Data Engineer по построению DWH,3–6 лет,Уровень дохода не указан,False,NaN,NaN,гибрид,Полная занятость,"[ETL, SQL, DWH, Apache Kafka, Debezium, BI, db...",3.0,"[PostgreSQL, Kafka, Debezium, dbt, Airflow, Fi...","[etl, sql, dwh, apache kafka, debezium, bi, db..."
69,65,https://hh.ru/vacancy/130308940,Senior Data Scientist\nВ архиве с 12 марта 202...,СОЗДАВАЙ ИННОВАЦИИ В АТМОСФЕРЕ СВОБОДЫ —\nИ РА...,2026-03-17 22:38:38.542,2026-03-17 22:38:38.542,"[Senior Data Scientist, Уровень дохода не указ...",Senior Data Scientist,1–3 года,Уровень дохода не указан,False,NaN,NaN,NaN,Полная занятость,[],1.0,"[Python, ML-библиотеки, SQL]","[python, ml-библиотеки, sql]"


In [230]:
def clean_text_vacancy(text):
    text = text.lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-zа-я0-9+#.\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [231]:
df_vac["text_clean"] = (
    df_vac["wrapper_text"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

In [232]:
df_vac

,id,vacancy_url,header_text,wrapper_text,created_at,updated_at,header_text_list,vacancy_name,experience,salary,salary_exists,salary_from,salary_to,work_format,employment_type,skills_list,experience_years_min,skills_llm,skills_final,text_clean
16,15,https://hh.ru/vacancy/129592277,Аналитик данных / экономист инвестиционных про...,46\nорганов исполнительной власти\n132\nтеррит...,2026-03-17 22:02:49.589,2026-03-17 22:02:49.589,[Аналитик данных / экономист инвестиционных пр...,Аналитик данных / экономист инвестиционных про...,1–3 года,Уровень дохода не указан,False,NaN,NaN,на месте работодателя,Полная занятость,[],1.0,"[Python, Django, Pandas, MS Office (Power Poin...","[python, django, pandas, ms office (power poin...",46 органов исполнительной власти 132 территори...
34,34,https://hh.ru/vacancy/129853395,Data Engineer (разработчик DWH)\nВ архиве с 17...,Задаем тренды в технологиях ритейла\nX5 Group ...,2026-03-17 22:13:18.663,2026-03-17 22:13:18.663,"[Data Engineer (разработчик DWH), Уровень дохо...",Data Engineer (разработчик DWH),3–6 лет,Уровень дохода не указан,False,NaN,NaN,"на месте работодателя, удалённо или гибрид",Полная занятость,"[SQL, Python, Big Data, Apache Airflow, Apache...",3.0,"[Spark, Flink, ClickHouse, Kafka, Trino, Apach...","[sql, python, big data, apache airflow, apache...",задаем тренды в технологиях ритейла x5 group —...
47,92,https://hh.ru/vacancy/129525525,Аналитик данных\nВ архиве с 15 февраля 2026\nУ...,Центр Компетенций Бизнес аналитики и финансово...,2026-03-17 22:56:00.590,2026-03-17 22:56:00.590,"[Аналитик данных, Уровень дохода не указан, Оп...",Аналитик данных,1–3 года,Уровень дохода не указан,False,NaN,NaN,на месте работодателя,Полная занятость,[],1.0,"[SQL, Python, Hadoop, Excel]","[sql, python, hadoop, excel]",центр компетенций бизнес аналитики и финансово...
64,162,https://hh.ru/vacancy/130038332,Data Engineer по построению DWH\nУровень доход...,Компания EcoFinance развивает и внедряет проду...,2026-03-17 23:43:51.117,2026-03-17 23:43:51.117,"[Data Engineer по построению DWH, Уровень дохо...",Data Engineer по построению DWH,3–6 лет,Уровень дохода не указан,False,NaN,NaN,гибрид,Полная занятость,"[ETL, SQL, DWH, Apache Kafka, Debezium, BI, db...",3.0,"[PostgreSQL, Kafka, Debezium, dbt, Airflow, Fi...","[etl, sql, dwh, apache kafka, debezium, bi, db...",компания ecofinance развивает и внедряет проду...
69,65,https://hh.ru/vacancy/130308940,Senior Data Scientist\nВ архиве с 12 марта 202...,СОЗДАВАЙ ИННОВАЦИИ В АТМОСФЕРЕ СВОБОДЫ —\nИ РА...,2026-03-17 22:38:38.542,2026-03-17 22:38:38.542,"[Senior Data Scientist, Уровень дохода не указ...",Senior Data Scientist,1–3 года,Уровень дохода не указан,False,NaN,NaN,NaN,Полная занятость,[],1.0,"[Python, ML-библиотеки, SQL]","[python, ml-библиотеки, sql]",создавай инновации в атмосфере свободы — и рас...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16757,16757,https://hh.ru/vacancy/131551598,Senior Data Scientist\nУровень дохода не указа...,"Мы в поиске специалиста по Data Science, котор...",2026-04-22 14:34:30.769,2026-04-22 14:34:30.769,"[Senior Data Scientist, Уровень дохода не указ...",Senior Data Scientist,не требуется,Уровень дохода не указан,False,NaN,NaN,на месте работодателя,Полная занятость,[],NaN,"[Python, pytorch, pandas, numpy, SQLAlchemy, S...","[python, pytorch, pandas, numpy, sqlalchemy, s...","мы в поиске специалиста по data science, котор..."
16759,16759,https://hh.ru/vacancy/132015906,Стажер Data Engineer\nУровень дохода не указан...,Стажировка в T2 — это полноценная оплачиваемая...,2026-04-22 14:34:52.526,2026-04-22 14:34:52.526,"[Стажер Data Engineer, Уровень дохода не указа...",Стажер Data Engineer,не требуется,Уровень дохода не указан,False,NaN,NaN,на месте работодателя или гибрид,Полная занятость,[],NaN,"[SQL, Python]","[sql, python]",стажировка в t2 — это полноценная оплачиваемая...
16760,16760,https://hh.ru/vacancy/131675291,Стажер Data Engineer\nУровень дохода не указан...,ДАВАЙ С НАМИ\nМы я

In [233]:
df_vac.to_csv("df_vac_new.csv", index=False)

## EDA по полученным датасетам

Для baseline оставим только столбцы, необходимые для сопоставления

In [245]:
df_rez.head()

,id,resume_url,header_text,wrapper_text,created_at,updated_at,resume_title,skills_list,specializations_list,experience_text,citizenship,text_clean
1,2,https://hh.ru/resume/c0d0ac9b0002b4446d0039ed1...,"Был меньше недели назад\nКандидат\nМужчина, 32...",Аналитик данных\nСпециализации:\nАналитик\nТип...,2026-02-23 17:14:55.601 +0300,2026-02-23 17:14:55.601 +0300,Аналитик данных,"[пользователь пк, ms sql, ms office, driving l...",[Аналитик],8.92,Россия,сентябрь 2024 сентябрь 2025 кофемания москва г...
2,3,https://hh.ru/resume/a28fcb6700084b93170039ed1...,"Была меньше недели назад\nКандидат\nЖенщина, 3...",Аналитик данных\n300 000 ₽ на руки\nСпециализа...,2026-02-23 17:15:11.186 +0300,2026-02-23 17:15:11.186 +0300,Аналитик данных,"[ms powerpoint, python, numpy, sql, git, pycha...","[BI-аналитик, аналитик данных, Аналитик, Бизне...",6.50,Россия,апрель 2025 по настоящее время termoland услуг...
3,4,https://hh.ru/resume/b1b0872200090d4ab70039ed1...,"Был меньше недели назад\nКандидат\nМужчина, 24...",Аналитик данных\n180 000 ₽ на руки\nСпециализа...,2026-02-23 17:15:27.759 +0300,2026-02-23 17:15:27.759 +0300,Аналитик данных,"[data science, регрессионный анализ, проверка ...",[Аналитик],2.67,Россия,декабрь 2024 по настоящее время центральный ба...
5,6,https://hh.ru/resume/989115340003c2cb260039ed1...,"Был сегодня в 12:23\nКандидат\nМужчина, 35 лет...",Аналитик данных\n150 000 ₽ на руки\nСпециализа...,2026-02-23 17:15:55.201 +0300,2026-02-23 17:15:55.201 +0300,Аналитик данных,"[sql, power bi, ms power bi, dax, tableau, abc...","[BI-аналитик, аналитик данных, Аналитик, Бизне...",NaN,Россия,январь 2025 по настоящее время нева дельта спб...
7,8,https://hh.ru/resume/29c348910002487e500039ed1...,"Был сегодня в 12:54\nКандидат\nМужчина, 39 лет...","Аналитик данных\nСпециализации:\nBI-аналитик, ...",2026-02-23 17:16:25.280 +0300,2026-02-23 17:16:25.280 +0300,Аналитик данных,"[python, vba, sql, ms office, oracle, qlik sen...","[BI-аналитик, аналитик данных]",14.42,Россия,октябрь 2011 по настоящее время сбер москва ra...


In [246]:
df_vac.head()

,id,vacancy_url,header_text,wrapper_text,created_at,updated_at,header_text_list,vacancy_name,experience,salary,salary_exists,salary_from,salary_to,work_format,employment_type,skills_list,experience_years_min,skills_llm,skills_final,text_clean
16,15,https://hh.ru/vacancy/129592277,Аналитик данных / экономист инвестиционных про...,46\nорганов исполнительной власти\n132\nтеррит...,2026-03-17 22:02:49.589,2026-03-17 22:02:49.589,[Аналитик данных / экономист инвестиционных пр...,Аналитик данных / экономист инвестиционных про...,1–3 года,Уровень дохода не указан,False,NaN,NaN,на месте работодателя,Полная занятость,[],1.0,"[Python, Django, Pandas, MS Office (Power Poin...","[python, django, pandas, ms office (power poin...",46 органов исполнительной власти 132 территори...
34,34,https://hh.ru/vacancy/129853395,Data Engineer (разработчик DWH)\nВ архиве с 17...,Задаем тренды в технологиях ритейла\nX5 Group ...,2026-03-17 22:13:18.663,2026-03-17 22:13:18.663,"[Data Engineer (разработчик DWH), Уровень дохо...",Data Engineer (разработчик DWH),3–6 лет,Уровень дохода не указан,False,NaN,NaN,"на месте работодателя, удалённо или гибрид",Полная занятость,"[SQL, Python, Big Data, Apache Airflow, Apache...",3.0,"[Spark, Flink, ClickHouse, Kafka, Trino, Apach...","[sql, python, big data, apache airflow, apache...",задаем тренды в технологиях ритейла x5 group —...
47,92,https://hh.ru/vacancy/129525525,Аналитик данных\nВ архиве с 15 февраля 2026\nУ...,Центр Компетенций Бизнес аналитики и финансово...,2026-03-17 22:56:00.590,2026-03-17 22:56:00.590,"[Аналитик данных, Уровень дохода не указан, Оп...",Аналитик данных,1–3 года,Уровень дохода не указан,False,NaN,NaN,на месте работодателя,Полная занятость,[],1.0,"[SQL, Python, Hadoop, Excel]","[sql, python, hadoop, excel]",центр компетенций бизнес аналитики и финансово...
64,162,https://hh.ru/vacancy/130038332,Data Engineer по построению DWH\nУровень доход...,Компания EcoFinance развивает и внедряет проду...,2026-03-17 23:43:51.117,2026-03-17 23:43:51.117,"[Data Engineer по построению DWH, Уровень дохо...",Data Engineer по построению DWH,3–6 лет,Уровень дохода не указан,False,NaN,NaN,гибрид,Полная занятость,"[ETL, SQL, DWH, Apache Kafka, Debezium, BI, db...",3.0,"[PostgreSQL, Kafka, Debezium, dbt, Airflow, Fi...","[etl, sql, dwh, apache kafka, debezium, bi, db...",компания ecofinance развивает и внедряет проду...
69,65,https://hh.ru/vacancy/130308940,Senior Data Scientist\nВ архиве с 12 марта 202...,СОЗДАВАЙ ИННОВАЦИИ В АТМОСФЕРЕ СВОБОДЫ —\nИ РА...,2026-03-17 22:38:38.542,2026-03-17 22:38:38.542,"[Senior Data Scientist, Уровень дохода не указ...",Senior Data Scientist,1–3 года,Уровень дохода не указан,False,NaN,NaN,NaN,Полная занятость,[],1.0,"[Python, ML-библиотеки, SQL]","[python, ml-библиотеки, sql]",создавай инновации в атмосфере свободы — и рас...


Распределение длин очищенных текстов

In [252]:
df_rez["text_len"] = df_rez["text_clean"].str.len()
df_vac["text_len"] = df_vac["text_clean"].str.len()

df_rez["text_len"].describe()

count    11940.000000
mean      3975.044389
std       2915.970838
min        136.000000
25%       2173.000000
50%       3306.000000
75%       4997.000000
max      33233.000000
Name: text_len, dtype: float64

In [253]:
df_vac["text_len"].describe()

count    1465.00000
mean     2656.03413
std       976.07746
min       577.00000
25%      2002.00000
50%      2493.00000
75%      3178.00000
max      7811.00000
Name: text_len, dtype: float64

Посмотрим на частотные слова (возможно, это поможет выявить "мусор")

In [248]:
all_words = " ".join(df_rez["text_clean"]).split()
Counter(all_words).most_common(20)

[('и', 209331),
 ('в', 104593),
 ('с', 69722),
 ('данных', 65819),
 ('по', 65602),
 ('на', 56823),
 ('для', 54373),
 ('data', 39374),
 ('and', 37791),
 ('анализ', 37057),
 ('of', 31757),
 ('образование', 29534),
 ('sql', 29066),
 ('высшее', 28233),
 ('python', 26160),
 ('разработка', 24082),
 ('работа', 22288),
 ('the', 22280),
 ('.', 21878),
 ('москва', 18468)]

N-граммы

In [249]:
vec = CountVectorizer(ngram_range=(2,2), max_features=20)
X = vec.fit_transform(df_vac["text_clean"])
vec.get_feature_names_out()

array(['power bi', 'анализа данных', 'будет плюсом', 'вакансия открыта',
       'где предстоит', 'график работы', 'ключевые навыки',
       'место работы', 'мы предлагаем', 'на вакансию', 'на основе',
       'оплата труда', 'опыт работы', 'от лет', 'предстоит работать',
       'работать москва', 'условия использования', 'формат работы',
       'что мы', 'яндекс условия'], dtype=object)

Распределение по навыкам

In [254]:
df_vac["skills_count"] = df_vac["skills_final"].apply(len)
df_rez["skills_count"] = df_rez["skills_list"].apply(len)

df_rez["skills_count"].describe()

count    11940.000000
mean        15.889866
std          9.827587
min          0.000000
25%          9.000000
50%         16.000000
75%         24.000000
max         65.000000
Name: skills_count, dtype: float64

In [255]:
df_vac["skills_count"].describe()

count    1465.000000
mean        9.232082
std         6.664331
min         0.000000
25%         4.000000
50%         8.000000
75%        13.000000
max        44.000000
Name: skills_count, dtype: float64

Самые популярные навыки

In [257]:
all_skills = sum(df_vac["skills_final"], [])
Counter(all_skills).most_common(100)

[('python', 1017),
 ('sql', 931),
 ('postgresql', 269),
 ('pandas', 263),
 ('clickhouse', 245),
 ('power bi', 231),
 ('numpy', 196),
 ('airflow', 193),
 ('pytorch', 178),
 ('задайте вопрос работодателю', 162),
 ('он получит его с откликом на вакансию', 162),
 ('где располагается место работы?', 162),
 ('какой график работы?', 162),
 ('вакансия открыта?', 162),
 ('какая оплата труда?', 162),
 ('как с вами связаться?', 162),
 ('другой вопрос', 162),
 ('etl', 139),
 ('apache airflow', 137),
 ('git', 135),
 ('docker', 135),
 ('scikit-learn', 129),
 ('анализ данных', 124),
 ('hadoop', 118),
 ('greenplum', 118),
 ('tableau', 117),
 ('spark', 115),
 ('ms excel', 86),
 ('pyspark', 83),
 ('kafka', 82),
 ('аналитическое мышление', 82),
 ('dwh', 72),
 ('tensorflow', 70),
 ('mysql', 70),
 ('ms sql', 69),
 ('dbt', 65),
 ('big data', 61),
 ('excel', 61),
 ('математическая статистика', 61),
 ('oracle', 57),
 ('catboost', 55),
 ('mlflow', 54),
 ('kubernetes', 54),
 ('matplotlib', 54),
 ('superset', 53

Также проверим, остались ли вакансии с отсутствием навыком после ручной обработки и обработки LLLM (если остались - удалим их)

In [267]:
(df_vac['skills_final'].apply(lambda x: len(x) == 0)).sum()

np.int64(22)

In [ ]:
df_vac = df_vac[df_vac['skills_final'].apply(lambda x: len(x) > 0)].copy()

### Удаление технических столбцов

Для baseline оставим только столбцы, необходимые для сопоставления

In [236]:
df_rez_base = df_rez[['id', 'resume_title', 'skills_list', 'experience_text', 'wrapper_text']]
df_vac_base = df_vac[['id', 'vacancy_name', 'skills_list', 'experience_years_min', 'wrapper_text']]

### Базовые характеристики

In [237]:
display(df_rez_base.dtypes.reset_index().rename(columns={'index': 'column', 0: 'dtype'}))
display(df_vac_base.dtypes.reset_index().rename(columns={'index': 'column', 0: 'dtype'}))

,column,dtype
0,id,int64
1,resume_title,object
2,skills_list,object
3,experience_text,float64
4,wrapper_text,str


,column,dtype
0,id,int64
1,vacancy_name,str
2,skills_list,object
3,experience_years_min,float64
4,wrapper_text,str


### Проверка и обработка пропусков

In [238]:
df_rez_base.isna().sum().sort_values(ascending=False)

experience_text    2599
id                    0
resume_title          0
skills_list           0
wrapper_text          0
dtype: int64

In [239]:
df_vac_base.isna().sum().sort_values(ascending=False)

experience_years_min    88
id                       0
vacancy_name             0
skills_list              0
wrapper_text             0
dtype: int64

In [269]:
df_rez_base[df_rez_base.isna().any(axis=1)]

,id,resume_title,skills_list,experience_text,wrapper_text


Видим пропуски в опыте работы

Логично считать, что кандидаты не заполняют опыт работы при его отсутствии, поэтому заполним нулями

In [241]:
df_rez_base['experience_text'] = df_rez_base['experience_text'].fillna(0)

In [268]:
df_vac_base[df_vac_base.isna().any(axis=1)]

,id,vacancy_name,skills_list,experience_years_min,wrapper_text


Видим 1 строку по вакансиям с неуказанным опытом работы

Пока заполним его нулем

In [243]:
df_vac_base['experience_years_min'] = df_vac_base['experience_years_min'].fillna(0)